# Sports Betting Analysis

Edge calculation, CLV tracking, pattern flagging.

In [ ]:
import numpy as np
from ipywidgets import interactive, HBox, VBox, Output, FloatSlider, IntSlider
from IPython.display import display

from speculator_ev_engine.sports.edge import compute_edge, compute_clv, CLVTracker
from speculator_ev_engine.ui.core.charts import clv_distribution, clv_over_time, pattern_flags_small_multiples
from speculator_ev_engine.ui.core.formatters import fmt_ev, fmt_pct

In [ ]:
# Simulate a CLV tracker with 200 bets
tracker = CLVTracker()
rng = np.random.default_rng(42)
sports = ['nfl', 'nba', 'mlb', 'nhl']
books = ['pinnacle', 'draftkings', 'fanduel']
for _ in range(200):
    sport = rng.choice(sports)
    book = rng.choice(books)
    open_o = rng.choice([-110, -105, +100, +105, +110, +120, +150, -130])
    close_shift = rng.normal(-10 if sport == 'nfl' else 0, 8)
    close_o = open_o + int(close_shift)
    if close_o == 0: close_o = -101 if open_o < 0 else 101
    tracker.add_record(sport, book, 'moneyline', 'medium', open_o, close_o)

clv_vals = np.array([float(r['clv']) for r in tracker.records])
fig = clv_distribution(clv_vals, float(np.mean(clv_vals)), float(np.median(clv_vals)))
display(fig)
plt.close(fig)

In [ ]:
# Pattern flags
flags = tracker.flag_patterns(min_samples=10)
flag_by_group = {}
for f in flags:
    g = f['group']
    flag_by_group.setdefault(g, []).append(f)
fig2 = pattern_flags_small_multiples(flag_by_group, metric_key='mean_clv')
display(fig2)
plt.close(fig2)